# 03 · Final evaluation, inference timing and export
Evaluate fixed checkpoints on held-out images. Thresholds are selected on validation
images only. Bootstrap intervals resample images, since source-video groups are unavailable. No invented results
are included: every table below is produced from your checkpoints and data.


This PNG dataset has no supplied video/source grouping. All video-named metrics, validation scores, thresholds and bootstrap groups below operate on individual images. They must be reported as image-level results, not source-video results. Existing folder splits are preserved.


## Runtime and embedded project setup
For training, select **Runtime → Change runtime type → GPU**. This cell unpacks
the complete project source into `/kaggle/working/freqblender_project` and installs support
dependencies. It keeps Kaggle's matched PyTorch/torchvision CUDA installation.
No GitHub fork or separate project ZIP upload is needed.


## Kaggle input and writable outputs
Attach `delno002/split-ffpp` to every notebook. Its root must contain
train/real, train/fake, val/real, val/fake, test/real and test/fake PNG folders.
Existing splits are preserved. Images are read directly from the attached dataset.
Save notebook outputs before ending a session. For later stages, attach the previous
saved output and set PREVIOUS_RUNS to its `freqblender/runs` directory.


In [ ]:
import shutil
import json, sys, subprocess
from pathlib import Path
import torch


FFPP_ROOT = Path("/kaggle/input/split-ffpp")  # EDIT if Kaggle mounts a different path
DRIVE_PROJECT = Path("/kaggle/working/freqblender")  # legacy variable name, local Kaggle path
DATA_ROOT = DRIVE_PROJECT / "prepared"
RUN_ROOT = DRIVE_PROJECT / "runs"
MANIFEST = DATA_ROOT / "manifest.csv"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
PREVIOUS_RUNS = None  # e.g. Path("/kaggle/input/previous-output/freqblender/runs")
if PREVIOUS_RUNS is not None:
    # Copy missing files only, preserving newer checkpoints from this session.
    previous = Path(PREVIOUS_RUNS)
    if not previous.is_dir():
        raise FileNotFoundError(previous)
    for source in previous.rglob("*"):
        target = RUN_ROOT / source.relative_to(previous)
        if source.is_file() and not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)
from freqblender.data import build_image_manifest
MANIFEST = build_image_manifest(FFPP_ROOT, MANIFEST)
versions = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(DRIVE_PROJECT / "environment.txt").write_text(versions)
print("Manifest:", MANIFEST)
print("Using image-level samples; original video/source identities are unavailable.")


In [ ]:
from freqblender.data import read_manifest
table = read_manifest(MANIFEST)
RUN_NAMES = ["teacher_spatial", "teacher_freqblender", "student_ce", "student_kd",
             "student_structure", "student_proposed"]
AVAILABLE = [name for name in RUN_NAMES if (RUN_ROOT/name/"best.pt").exists()]
assert "teacher_freqblender" in AVAILABLE and "student_proposed" in AVAILABLE
print("Comparing:", AVAILABLE)
# Same hardware, input size and precision are required for a matched comparison.
configs = {name: torch.load(RUN_ROOT/name/"best.pt", map_location="cpu", weights_only=True)["config"]
           for name in AVAILABLE}
assert len({cfg["size"] for cfg in configs.values()}) == 1
assert len({cfg["manifest_sha256"] for cfg in configs.values()}) == 1


## Evaluate each model in a separate process
Each process holds only one detector, making peak GPU-memory comparisons meaningful.
Model-only timing includes normalization and forward execution, with CUDA warmup and
synchronization. It excludes face detection, decoding, resizing and transfer. All runs
use FP32 at batch sizes 1 and 16; change 16 consistently if GPU memory requires it.


In [ ]:
for name in AVAILABLE:
    subprocess.run([sys.executable, "-m", "freqblender.evaluate",
                    "--checkpoint", str(RUN_ROOT/name/"best.pt"),
                    "--manifest", str(MANIFEST), "--output", str(RUN_ROOT/name/"evaluation"),
                    "--batch-size", "16", "--bootstrap", "1000"], check=True)


In [ ]:
import pandas as pd
reports, rows = {}, []
for name in AVAILABLE:
    report = json.loads((RUN_ROOT/name/"evaluation/metrics.json").read_text())
    reports[name] = report
    single = report["benchmark"][0]
    throughput = report["benchmark"][1]
    for dataset, result in report["datasets"].items():
        rows.append(dict(run=name, dataset=dataset, image_auc=result["video"]["auc"],
                         frame_auc=result["frame"]["auc"], auc_ci_low=result["video_auc_ci"]["low"],
                         auc_ci_high=result["video_auc_ci"]["high"],
                         balanced_accuracy=result["video"]["balanced_accuracy"],
                         parameters=single["parameters"], batch1_ms=single["median_batch_ms"],
                         batch1_p95_ms=single["p95_batch_ms"], throughput=throughput["images_per_second"],
                         peak_cuda_mb=throughput["peak_cuda_mb"]))
results = pd.DataFrame(rows)
teacher_ms = reports["teacher_freqblender"]["benchmark"][0]["median_batch_ms"]
results["speedup_vs_teacher"] = teacher_ms/results.batch1_ms
results.to_csv(RUN_ROOT/"comparison.csv", index=False)
display(results)


## Paired AUC differences against ordinary KD
The proposed loss should be judged against the same compact architecture with
ordinary KD, not only against the larger teacher. These intervals reuse each
sampled image for both models. An interval crossing zero does not establish
an AUC gain. This accounts for test-set sampling, not training-seed variation.


In [ ]:
from freqblender.evaluate import paired_auc_difference
if "student_kd" in AVAILABLE:
    proposed = pd.read_csv(RUN_ROOT/"student_proposed/evaluation/test_video_predictions.csv")
    reference = pd.read_csv(RUN_ROOT/"student_kd/evaluation/test_video_predictions.csv")
    differences = {}
    for dataset in proposed.dataset.unique():
        differences[dataset] = paired_auc_difference(
            proposed[proposed.dataset == dataset], reference[reference.dataset == dataset])
    display(pd.DataFrame(differences).T)
    (RUN_ROOT/"paired_auc_difference_vs_kd.json").write_text(json.dumps(differences, indent=2))


In [ ]:
import matplotlib.pyplot as plt
fig, axis = plt.subplots(figsize=(9, 5))
for _, row in results[results.dataset == "FF++"].iterrows():
    axis.scatter(row.batch1_ms, row.image_auc, s=70)
    axis.annotate(row.run, (row.batch1_ms, row.image_auc), xytext=(5, 5), textcoords="offset points", fontsize=8)
axis.set_xlabel("Median batch-1 model latency (ms, lower is better)")
axis.set_ylabel("Held-out image AUC (higher is better)")
axis.grid(alpha=.2)
fig.tight_layout()
fig.savefig(RUN_ROOT/"accuracy_latency.png", dpi=180)


## Export the compact detector
Export contains only the student and input normalization. No DCT, FPNet, auxiliary
network or teacher is needed. The sidecar describes input/output conventions.
Detection thresholds remain in each run's evaluation report.


In [ ]:
from freqblender.evaluate import export_detector
EXPORT = RUN_ROOT / "student_proposed" / "student.ts"
export_detector(RUN_ROOT/"student_proposed/best.pt", EXPORT)
print("Export:", EXPORT, "Size MB:", EXPORT.stat().st_size/2**20)


In [ ]:
import numpy as np
from PIL import Image
CROP_PATH = Path(table[table.split == "test"].iloc[0].path)  # replace with YOUR FACE CROP
size = configs["student_proposed"]["size"]
device = "cuda" if torch.cuda.is_available() else "cpu"
deployed = torch.jit.load(str(EXPORT), map_location=device).eval()
with Image.open(CROP_PATH) as image:
    image = image.convert("RGB").resize((size, size), Image.Resampling.BILINEAR)
    tensor = torch.from_numpy(np.array(image, dtype=np.float32)/255).permute(2,0,1)[None].to(device)
with torch.inference_mode():
    fake_score = deployed(tensor).softmax(1)[0,1].item()
threshold = reports["student_proposed"]["datasets"]["FF++"]["video"]["threshold"]
print({"fake_score": fake_score, "validation_image_threshold": threshold,
       "thresholded_prediction": "fake" if fake_score >= threshold else "real"})
# This threshold is calibrated on validation images, not videos.


For external-dataset testing, append its face-crop rows as `test` with a unique dataset
name; retain FF++ validation. Inspect `metrics.json` for AP, low-FPR TPR and timing
metadata. Compare multiple seeds and paired group-bootstrap differences before claiming
small gains. FF++ results alone do not establish cross-dataset generalization.
